# Agent Registry의 Metadata 검색

이 Notebook에서는 AWS Agent Registry의 semantic search를 사용하는 방법을 보여 줍니다.

## 시나리오

전자 상거래 회사의 개발자가 새로운 fulfillment 에이전트를 연결해야 합니다. 플랫폼에는 이미
MCP 도구와 A2A 에이전트가 배포되어 있지만 어떤 기능이 있으며 각 도구가 무엇을 할 수 있는지
정확히 아는 사람이 없습니다. 개발자는 내부 문서를 읽거나 동료에게 묻지 않고도 Registry의
자연어 검색을 사용하여 적합한 도구를 찾습니다.

## Registry의 도구

| 도구 | 프로토콜 | 용도 |
|---|---|---|
| `order_lookup_tool` | MCP | 주문 세부 정보 조회 및 고객별 주문 목록 확인 |
| `order_update_tool` | MCP | 주문 상태 또는 배송 주소 업데이트 |
| `order_cancel_tool` | MCP | 주문 취소 |
| `email_send_tool` | MCP | 고객에게 거래 이메일 전송 |
| `email_template_tool` | MCP | 재사용 가능한 이메일 template 관리 |
| `sms_notify_tool` | MCP | SMS 알림 전송 |
| `payment_status_tool` | MCP | 주문의 결제 상태 조회 |
| `inventory_check_tool` | MCP | 하나 이상의 SKU에 대한 가용 재고 확인 |
| `shipping_track_tool` | MCP | 배송 추적 및 배송 예정일 확인 |
| `returns_processing_tool` | MCP (remote) | 제품 반품 처리 및 반품 label 생성 |
| `loyalty_rewards_tool` | MCP (remote) | loyalty point 관리 및 reward 사용 |
| `payment_refund_tool` | A2A | 다단계 검증을 거쳐 환불 처리 |
| `inventory_reserve_tool` | A2A | rollback을 지원하는 재고 예약 |
| `shipping_update_tool` | A2A | 배송사 선택 및 상태 업데이트를 포함한 배송 생성 |

## 학습 목표

- 샘플 전자 상거래 도구를 Registry에 등록
- **자연어 query**와 함께 `SearchRegistryRecords`를 사용하여 도구 검색
- Metadata filter(`$eq`, `$ne`, `$in`, `$and`, `$or`)로 **결과 필터링**
- 레코드 세부 정보로 **drill-down**하여 연결 정보 추출
- 하나의 query가 여러 프로토콜을 반환하는 **유형 간 검색** 처리

## 아키텍처

<img src="consumer-discovery-semantic-search.png" alt="Consumer 검색 아키텍처" width="70%"/>


## Semantic Search 작동 방식

AWS Agent Registry는 의미 이해와 keyword matching을 결합한 **hybrid search**를 사용합니다.
`SearchRegistryRecords`를 호출하면 두 가지 검색이 병렬로 실행되고 결과가 순위가 매겨진
하나의 목록으로 병합됩니다.

- **Semantic search**는 query를 vector 표현으로 변환하여 vector화된 레코드 내용과 비교합니다.
  정확히 같은 단어가 레코드에 없어도 개념적으로 관련된 레코드를 찾습니다. 예를 들어
  "process a customer return" query는 `returns_processing_tool`이라는 레코드와 일치할 수 있습니다.
- **Keyword search**는 전통적인 keyword 관련도를 사용하여 query와 레코드 필드의 텍스트 내용을
  비교합니다. 정확한 이름 조회와 특정 기술 용어 검색에 효과적입니다.

결과는 두 검색 방식의 관련도를 결합하여 순위를 매깁니다. Semantic search와 keyword search에서
모두 높은 점수를 받은 레코드는 한 방식에서만 높은 점수를 받은 레코드보다 높은 순위를 얻습니다.
Keyword search에서는 레코드 **name**의 영향력이 가장 크며, 그다음으로 **description**과
**descriptor content**가 동일한 영향력을 갖습니다.

### Index에 포함되는 항목

Semantic search를 위해 다음을 포함한 전체 레코드를 vector화합니다.

- **Name** — Keyword matching에 사용됩니다. 명확하고 설명적인 이름은 검색 가능성을 높입니다.
- **Description** — Keyword 및 semantic matching에 모두 사용됩니다. 목적과 사용 사례를 설명하는
  자연어 description은 짧은 기술 label보다 검색하기 쉽습니다.
- **Descriptors** — 전체 프로토콜 정의(MCP server/tools, A2A agent card, custom JSON)가
  semantic matching에 사용됩니다. 여기에는 도구 이름과 설명, input parameter 이름 및 기능 요약이 포함됩니다.

### 효과적인 Query 작성

- **정확한 이름 조회** — 짧고 구체적인 query를 사용합니다: `"payment_status_tool"` 또는 `"order_lookup"`.
- **기능 탐색** — 자연어를 사용합니다: `"process a customer refund for a returned item"`.
  Semantic search가 의도를 이해하므로 일치하는 레코드에 정확히 같은 단어가 포함될 필요는 없습니다.
- **Query 텍스트에 filter를 섞지 않기** — `"find all MCP servers for payments"`와 같은 query는
  전체 문장을 semantic search에 전달합니다. 이때 "MCP servers"는 filter가 아닌 개념적 의도의
  일부로 해석됩니다. 대신 속성 제약 조건에는 metadata filter를 사용하고 query는 주제에 집중하세요.
  ```python
  dp_client.search_registry_records(
      registryIds=[REGISTRY_ARN],
      searchQuery="process a refund",
      filters={"descriptorType": {"$eq": "MCP"}}
  )
  ```

### 검색하기 쉬운 레코드 작성

- 리소스가 수행하는 작업과 해결하는 문제를 설명하는 description을 작성합니다.
- MCP 서버에 완전한 도구 정의를 제공합니다. 도구 description과 input parameter description이
  모두 검색 관련도에 영향을 줍니다.
- 사용자가 검색할 가능성이 높은 관련 keyword를 name과 description에 포함합니다.

**Approved** 상태의 레코드만 검색 결과에 표시됩니다. Draft, Pending Approval, Rejected 또는
Deprecated 상태의 레코드는 반환되지 않습니다.

## 설정

### 사전 요구 사항
- AgentCore 서비스를 지원하는 boto3
- 다음 권한이 있는 IAM 사용자 또는 role(`ACCOUNT_ID`와 리전은 필요에 따라 변경)

<details>
<summary>필수 IAM policy(클릭하여 펼치기)</summary>

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowCreateRegistry",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:CreateRegistry"],
            "Resource": ["arn:aws:bedrock-agentcore:us-west-2:ACCOUNT_ID:*"]
        },
        {
            "Sid": "AllowGetUpdateDeleteRegistry",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetRegistry",
                "bedrock-agentcore:DeleteRegistry"
            ],
            "Resource": ["arn:aws:bedrock-agentcore:us-west-2:ACCOUNT_ID:registry/*"]
        },
        {
            "Sid": "AllowCreateAndListRecords",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:CreateRegistryRecord",
                "bedrock-agentcore:SearchRegistryRecords"
            ],
            "Resource": ["arn:aws:bedrock-agentcore:us-west-2:ACCOUNT_ID:registry/*"]
        },
        {
            "Sid": "AllowRecordOperations",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetRegistryRecord",
                "bedrock-agentcore:DeleteRegistryRecord",
                "bedrock-agentcore:SubmitRegistryRecordForApproval"
            ],
            "Resource": ["arn:aws:bedrock-agentcore:us-west-2:ACCOUNT_ID:registry/*/record/*"]
        }
    ]
}
```

</details>

In [ ]:
!pip install -q boto3
!pip install --upgrade boto3

In [ ]:
import boto3
import json
import os
import time
from datetime import datetime

# 구성
AWS_REGION = "us-west-2"

# Amazon SageMaker Notebook을 사용하지 않는 경우 AWS 자격 증명 설정
os.environ["AWS_PROFILE"] = "your-profile-name"

# boto3 세션 생성
session = boto3.Session(region_name=AWS_REGION)

# 클라이언트 생성
cp_client = session.client("bedrock-agentcore-control")
dp_client = session.client("bedrock-agentcore")

print(f"Session ready | Region: {AWS_REGION}")

---
## 2. Registry에 샘플 데이터 등록

`autoApproval: True`로 새 Registry를 생성하고 `registry-records.json`에 정의된 14개의 샘플
도구(MCP stdio 9개 + MCP remote 2개 + A2A 에이전트 3개)를 등록합니다.

In [ ]:
# 자동 승인이 활성화된 Registry 생성
registry_name = f"consumerDiscovery_{datetime.now().strftime('%Y%m%d%H%M%S')}"

create_resp = cp_client.create_registry(
    name=registry_name,
    description="Registry for consumer discovery journey demo",
    approvalConfiguration={"autoApproval": True},
)

REGISTRY_ARN = create_resp["registryArn"]
REGISTRY_ID = REGISTRY_ARN.split("/")[-1]

print("Registry created!")
print(f"  ARN: {REGISTRY_ARN}")
print(f"  ID:  {REGISTRY_ID}")

# Registry가 READY 상태가 될 때까지 대기
while True:
    r = cp_client.get_registry(registryId=REGISTRY_ID)
    if r["status"] == "READY":
        print("  Status: READY")
        break
    print(f"  Status: {r['status']} - please wait...")
    time.sleep(10)

In [ ]:
# JSON 파일에서 샘플 레코드 로드
with open("registry-records.json", "r") as f:
    SEED_RECORDS = json.load(f)

print(f"Loaded {len(SEED_RECORDS)} seed records")

# 모든 Registry 레코드 생성
record_ids = []
for rec in SEED_RECORDS:
    resp = cp_client.create_registry_record(
        registryId=REGISTRY_ID,
        name=rec["name"],
        description=rec["description"],
        descriptorType=rec["protocol"],
        descriptors=rec["descriptors"],
        recordVersion=rec["recordVersion"],
    )
    record_id = resp["recordArn"].split("/")[-1]
    record_ids.append(record_id)
    print(f"  [{rec['protocol']:3s}] {rec['name']} -> {record_id}")

print(f"\nCreated {len(record_ids)} records.")

# 모든 레코드가 CREATING 상태를 벗어날 때까지 대기
print("Waiting for records to be ready...")
for rid in record_ids:
    while True:
        rec = cp_client.get_registry_record(registryId=REGISTRY_ID, recordId=rid)
        if rec["status"] != "CREATING":
            break
        time.sleep(2)
print("All records ready.")

# 각 레코드의 승인 요청 제출(autoApproval=True이므로 바로 APPROVED로 전환)
for rid in record_ids:
    cp_client.submit_registry_record_for_approval(registryId=REGISTRY_ID, recordId=rid)

print(f"Submitted {len(record_ids)} records for approval.")

# 검색 index 전파 대기
print("Waiting 45s for search index propagation...")
time.sleep(45)
print("Ready for discovery.")

---
## 3. Consumer 검색 과정

**Consumer 관점**으로 전환합니다. Consumer는 필요한 기능은 알지만 정확한 도구 이름은 모릅니다.
자연어 검색을 사용하여 적합한 도구를 찾습니다.

### Helper: 검색 함수

In [ ]:
def search_raw(query, max_results=10):
    """레지스트리를 검색하고 원본 JSON API 응답을 출력합니다."""
    response = dp_client.search_registry_records(registryIds=[REGISTRY_ARN], searchQuery=query, maxResults=max_results)
    response.pop("ResponseMetadata", None)
    print(json.dumps(response, indent=2, default=str))
    return response


def search(query, max_results=10, filters=None):
    """레지스트리를 검색해 결과를 보기 좋게 표시하고 선택적으로 메타데이터 필터를 적용합니다."""
    params = dict(registryIds=[REGISTRY_ARN], searchQuery=query, maxResults=max_results)
    if filters:
        params["filters"] = filters
    response = dp_client.search_registry_records(**params)
    records = response.get("registryRecords", [])
    header = f'Search: "{query}"'
    if filters:
        header += f"  |  Filter: {json.dumps(filters)}"
    print(header)
    print(f"Found {len(records)} result(s)\n")
    for rec in records:
        print(f"  [{rec['descriptorType']:3s}]  {rec['name']}")
        print(f"        {rec.get('description', 'N/A')}")
        print()
    return response


print("search() and search_raw() helpers ready.")

### `maxResults`로 결과 수 제어

`maxResults` 파라미터는 검색 호출당 반환되는 레코드 수를 제어합니다.
**1~20**의 값을 사용할 수 있으며 기본값은 **10**입니다. 위 helper 함수의 기본값도 10이지만
호출할 때마다 다른 값으로 지정할 수 있습니다.

가장 관련도 높은 결과만 필요할 때는 작은 값을 사용하고(예: "give me the best result"에는
`maxResults=1`), 넓은 도메인을 탐색할 때는 값을 늘립니다.

In [ ]:
# 관련도가 가장 높은 결과 3개만 반환
results = search("order management", max_results=3)

### 3.1 주문 관리

개발자는 fulfillment 에이전트에서 주문 작업을 처리해야 합니다.
넓은 범위로 검색하면 Registry가 MCP와 A2A의 모든 주문 관련 도구를 표시해야 합니다.

In [ ]:
results = search("I need to look up, update, and cancel customer orders")

### 3.2 고객 알림

개발자는 주문 이벤트를 고객에게 알려야 합니다. 이메일과 SMS 중 무엇이 필요한지 또는
둘 다 필요한지 확실하지 않으므로 Registry가 판단하도록 합니다.

In [ ]:
results = search("send notifications to customers via email or SMS")

### 3.3 결제 및 환불

개발자에게 결제 기능이 필요합니다. 이 query는 MCP `payment_status_tool`(간단한 상태 조회)과
A2A `payment_refund_tool`(다단계 환불 에이전트)을 모두 반환해야 합니다. 이를 통해 하나의 query가
여러 프로토콜의 도구를 표시하는 방식을 확인할 수 있습니다.

In [ ]:
results = search("check payment status or issue a refund for an order")

### 3.4 재고 관리

개발자에게 재고 기능이 필요합니다. MCP `inventory_check_tool`(읽기 전용 재고 조회)과
A2A `inventory_reserve_tool`(rollback을 지원하는 stateful 예약)을 모두 표시해야 합니다.

In [ ]:
results = search("check product availability and reserve stock for an order")

### 3.5 배송

개발자는 배송을 처리해야 합니다. 읽기 도구(`shipping_track_tool`)와
쓰기 도구(`shipping_update_tool`)를 모두 표시해야 합니다.

In [ ]:
results = search("track shipments and get delivery estimates")

### 3.6 Drill-Down: MCP 도구 연결 세부 정보

검색으로 도구를 찾은 후 consumer는 전체 레코드를 가져와 실제 호출에 필요한 연결 정보인
Gateway URL, transport type, 사용 가능한 도구 및 필수 자격 증명을 확인합니다.

In [ ]:
# 주문 조회 도구를 검색하고 첫 번째 MCP 결과를 자세히 확인
response = search("look up order details by order ID")
records = response.get("registryRecords", [])
mcp_hits = [r for r in records if r["descriptorType"] == "MCP"]

if mcp_hits:
    hit = mcp_hits[0]
    record_id = hit["recordId"]
    print(f"Drilling into: {hit['name']} ({record_id})\n")

    full = cp_client.get_registry_record(registryId=REGISTRY_ID, recordId=record_id)
    mcp = full.get("descriptors", {}).get("mcp", {})

    server = json.loads(mcp.get("server", {}).get("inlineContent", "{}"))
    tools = json.loads(mcp.get("tools", {}).get("inlineContent", "{}"))

    print("Server descriptor:")
    print(json.dumps(server, indent=2))
    print("\nTools descriptor:")
    print(json.dumps(tools, indent=2))
else:
    print("No MCP results found.")

### 3.7 Drill-Down: A2A 에이전트 세부 정보

이제 A2A 에이전트를 자세히 살펴보고 AgentCore Runtime을 통한 호출에 필요한 agent ARN과
선언된 기능을 추출합니다.

In [ ]:
# 환불 처리를 검색하고 A2A 결과를 자세히 확인
response = search("issue a refund for a completed order")
records = response.get("registryRecords", [])
a2a_hits = [r for r in records if r["descriptorType"] == "A2A"]

if a2a_hits:
    hit = a2a_hits[0]
    record_id = hit["recordId"]
    print(f"Drilling into: {hit['name']} ({record_id})\n")

    full = cp_client.get_registry_record(registryId=REGISTRY_ID, recordId=record_id)
    card = full.get("descriptors", {}).get("a2a", {}).get("agentCard", {})
    agent_card = json.loads(card.get("inlineContent", "{}"))

    print("Agent card:")
    print(json.dumps(agent_card, indent=2))
else:
    print("No A2A results found.")

### 3.8 유형 간 검색: 전체 Fulfillment 워크플로

주문 fulfillment에 대한 광범위한 query는 여러 프로토콜의 도구를 반환해야 합니다.
Stateless 읽기/쓰기에는 MCP 도구를, stateful 다단계 작업에는 A2A 에이전트를 반환합니다.
이를 통해 Registry가 완전한 에이전트 워크플로 구성에 어떻게 도움이 되는지 확인할 수 있습니다.

In [ ]:
response = search("fulfill an e-commerce order end to end — inventory, payment, and shipping")
records = response.get("registryRecords", [])

# 프로토콜별로 그룹화
by_protocol = {}
for rec in records:
    proto = rec["descriptorType"]
    by_protocol.setdefault(proto, []).append(rec["name"])

print("\nResults grouped by protocol:")
for proto, names in sorted(by_protocol.items()):
    print(f"  {proto}: {', '.join(names)}")

print(f"\nTotal: {len(records)} tools discovered across {len(by_protocol)} protocol(s)")

### 3.9 Inline Descriptor 내용으로 검색

전체 inline descriptor 내용을 포함한 레코드 전체가 semantic search를 위해 vector화됩니다.
따라서 레코드의 최상위 name이나 description에 용어가 없더라도 도구 이름, input parameter 이름,
A2A skill identifier처럼 프로토콜 정의 내부에만 있는 용어를 검색하여 도구를 찾을 수 있습니다.

In [ ]:
# MCP tools descriptor 내부에 있는 도구 이름으로 검색
results = search("check_inventory")

In [ ]:
# inputSchema에만 있는 input parameter 이름("sku")으로 검색
results = search("SKU stock availability")

In [ ]:
# Agent card descriptor 내부에 있는 skill로 검색
results = search("assign-carrier")

### 3.10 일치 결과가 없는 검색

일치하는 도구가 없으면 어떻게 될까요? Consumer에게 빈 결과 집합이 표시됩니다. Registry는
관련 없는 결과 대신 올바르게 아무것도 반환하지 않습니다.

In [ ]:
results = search("quantum computing simulation, molecular dynamics")

### 3.11 원본 API 응답

프로그래밍 방식으로 사용할 수 있도록 `search_raw()`가 API의 전체 JSON 응답을 반환하고 출력합니다.

In [ ]:
raw_response = search_raw("order lookup")

### 3.12 Metadata Filter를 적용한 검색

Registry는 자연어 query 외에도 semantic search와 결합할 수 있는 **구조화된 metadata filter**를
지원합니다. Filter는 field 수준 연산자(`$eq`, `$ne`, `$in`)와 논리 연산자(`$and`, `$or`)가 있는
MongoDB 스타일 구문을 사용합니다.

Filter 적용 가능 field: `name`, `descriptorType`, `version`

#### Descriptor type으로 필터링(`$eq`)

A2A 에이전트를 제외하고 MCP 도구만 반환합니다.

In [ ]:
results = search("payment", filters={"descriptorType": {"$eq": "MCP"}})

#### Descriptor type 제외(`$ne`)

MCP 도구를 제외한 모든 항목을 반환합니다.

In [ ]:
results = search("shipping inventory refund", filters={"descriptorType": {"$ne": "MCP"}})

#### 여러 Descriptor type 일치(`$in`)

MCP 또는 A2A인 레코드를 반환합니다.

In [ ]:
results = search("order management", filters={"descriptorType": {"$in": ["MCP", "A2A"]}})

#### Version으로 필터링(`$eq`)

특정 version의 레코드만 반환합니다.

In [ ]:
results = search("order", filters={"version": {"$eq": "1.0"}})

#### `$and`로 조건 결합

Version 1.0의 MCP 도구만 반환합니다.

In [ ]:
results = search(
    "email notification",
    filters={"$and": [{"descriptorType": {"$eq": "MCP"}}, {"version": {"$eq": "1.0"}}]},
)

#### `$or`로 하나 이상의 조건 일치

MCP 또는 A2A인 레코드를 반환합니다. 단일 field에서는 `$in`과 같지만,
`$or`는 서로 다른 field의 조건을 결합할 수 있습니다.

In [ ]:
results = search(
    "inventory",
    filters={"$or": [{"descriptorType": {"$eq": "MCP"}}, {"descriptorType": {"$eq": "A2A"}}]},
)

#### 정확한 이름으로 필터링(`$eq`)

결과를 특정 레코드 이름으로 제한합니다.

In [ ]:
results = search("payment", filters={"name": {"$eq": "payment_status_tool"}})

---
## 4. 정리

이 Notebook에서 생성한 모든 레코드와 Registry를 삭제합니다.

In [ ]:
# 모든 레코드 삭제
for rid in record_ids:
    try:
        cp_client.delete_registry_record(registryId=REGISTRY_ID, recordId=rid)
        print(f"  Deleted record: {rid}")
    except Exception as e:
        print(f"  Error deleting {rid}: {e}")

# Registry 자체 삭제
try:
    cp_client.delete_registry(registryId=REGISTRY_ID)
    print(f"  Deleted registry: {REGISTRY_ID}")
except Exception as e:
    print(f"  Error deleting registry: {e}")

print("\nCleanup complete!")